# FLEXIMOD market result quick analysis

Small notebook for checking market electricity procurement and market cost/revenue components.

Expected result files in `RESULTS_DIR`:

- `market_ledger.csv` preferred for market positions and settlement values
- `dispatch_results.csv` used as fallback
- `summary_indicators.csv` optional


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

# Change this to the output folder of the case you want to analyse.
RESULTS_DIR = Path("../data/output/steel_plant_DE_steel_cost_minimization")

RESULTS_DIR.resolve()

In [ ]:
def read_optional_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path.name}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    print(f"Loaded {path.name}: {len(df):,} rows, {len(df.columns):,} columns")
    return df


market = read_optional_csv(RESULTS_DIR / "market_ledger.csv")
dispatch = read_optional_csv(RESULTS_DIR / "dispatch_results.csv")
summary = read_optional_csv(RESULTS_DIR / "summary_indicators.csv")

market.head() if not market.empty else dispatch.head()

In [ ]:
def numeric_series(df: pd.DataFrame, column: str, default: float = 0.0) -> pd.Series:
    if df.empty or column not in df.columns:
        return pd.Series([default], dtype=float)
    return pd.to_numeric(df[column], errors="coerce").fillna(default)


def column_sum(df: pd.DataFrame, column: str) -> float:
    return float(numeric_series(df, column).sum())


def first_existing(df: pd.DataFrame, columns: list[str]) -> str | None:
    for column in columns:
        if not df.empty and column in df.columns:
            return column
    return None


def weighted_value(df: pd.DataFrame, quantity_col: str, price_col: str) -> float:
    if df.empty or quantity_col not in df.columns or price_col not in df.columns:
        return 0.0
    quantity = pd.to_numeric(df[quantity_col], errors="coerce").fillna(0.0)
    price = pd.to_numeric(df[price_col], errors="coerce").fillna(0.0)
    return float((quantity * price).sum())

## Electricity procurement by market

This plot answers: how much electricity came from day-ahead, intraday buy/sell, and activated aFRR down energy?

In [ ]:
source = market if not market.empty else dispatch

procurement = {
    "Day-ahead procurement": column_sum(
        source,
        first_existing(source, ["day_ahead_position_MWh_el", "DA_position_MWh"]) or "",
    ),
    "Intraday buy": column_sum(
        source,
        first_existing(source, ["intraday_buy_MWh_el", "IDC_buy_MWh"]) or "",
    ),
    "Intraday sell": -column_sum(
        source,
        first_existing(source, ["intraday_sell_MWh_el", "IDC_sell_MWh"]) or "",
    ),
    "aFRR activated energy": column_sum(
        source,
        first_existing(source, ["afrr_energy_activated_MWh_el", "afrr_energy_activated_MWh"]) or "",
    ),
}

procurement_df = pd.DataFrame(
    {"market": procurement.keys(), "electricity_MWh": procurement.values()}
).query("electricity_MWh != 0")

display(procurement_df)

if procurement_df.empty:
    print("No electricity procurement or aFRR activation is recorded in these results.")
else:
    ax = procurement_df.plot.bar(
        x="market",
        y="electricity_MWh",
        legend=False,
        color=["#4C78A8", "#72B7B2", "#F58518", "#B279A2"][: len(procurement_df)],
        figsize=(8, 4),
    )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("Electricity [MWh_el]")
    ax.set_xlabel("")
    ax.set_title("Electricity procurement by market")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

## Market cost / revenue waterfall

Positive bars increase cost. Negative bars are revenues or sales that reduce net market cost.

The waterfall focuses on market-facing electricity settlement, not material input costs such as iron ore or lime.

In [ ]:
if not market.empty:
    components = {
        "Day-ahead electricity": weighted_value(
            market,
            "day_ahead_position_MWh_el",
            "day_ahead_delivered_price_EUR_per_MWh_el",
        ),
        "Intraday buy": weighted_value(
            market,
            "intraday_buy_MWh_el",
            "intraday_delivered_price_EUR_per_MWh_el",
        ),
        "Intraday sell": -weighted_value(
            market,
            "intraday_sell_MWh_el",
            "intraday_delivered_price_EUR_per_MWh_el",
        ),
        "aFRR energy settlement": weighted_value(
            market,
            "afrr_energy_activated_MWh_el",
            "afrr_energy_delivered_price_EUR_per_MWh_el",
        ),
        "aFRR capacity revenue": -column_sum(market, "afrr_capacity_revenue_EUR"),
    }
else:
    components = {
        "Electricity market cost": column_sum(dispatch, "electricity_market_cost_EUR"),
        "Additional electricity charges": column_sum(
            dispatch, "additional_electricity_charges_cost_EUR"
        ),
        "aFRR capacity revenue": -column_sum(dispatch, "afrr_capacity_revenue_EUR"),
    }

waterfall_df = pd.DataFrame(
    {"component": components.keys(), "value_EUR": components.values()}
).query("value_EUR != 0")
waterfall_df.loc[len(waterfall_df)] = ["Net market cost", waterfall_df["value_EUR"].sum()]

display(waterfall_df)

running = waterfall_df["value_EUR"].iloc[:-1].cumsum().shift(fill_value=0.0)
values = waterfall_df["value_EUR"].iloc[:-1]
labels = waterfall_df["component"].iloc[:-1].tolist()
net = waterfall_df["value_EUR"].iloc[-1]

fig, ax = plt.subplots(figsize=(10, 4.8))
colors = ["#D95F02" if value >= 0 else "#1B9E77" for value in values]
ax.bar(labels, values, bottom=running, color=colors)
ax.bar(["Net market cost"], [net], color="#4C78A8")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("EUR")
ax.set_title("Market cost and revenue waterfall")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

## Optional: per-plant market procurement

Useful when the output contains several plants/routes.

In [ ]:
if not market.empty and "plant_name" in market.columns:
    per_plant = market.groupby("plant_name", dropna=False).agg(
        day_ahead_MWh=("day_ahead_position_MWh_el", "sum"),
        afrr_activated_MWh=("afrr_energy_activated_MWh_el", "sum"),
        capacity_revenue_EUR=("afrr_capacity_revenue_EUR", "sum"),
    )
    display(per_plant)
    per_plant[["day_ahead_MWh", "afrr_activated_MWh"]].plot.bar(
        stacked=True,
        figsize=(10, 4),
        color=["#4C78A8", "#B279A2"],
    )
    plt.ylabel("Electricity [MWh_el]")
    plt.title("Electricity procurement by plant")
    plt.tight_layout()
else:
    print("No market_ledger.csv with plant_name column available.")

## Net cost by plant

Scans every use-case output folder under `data/output`, calculates net cost for every plant, and adds a portfolio total. It uses the most complete available output: net operating cost including grid fees, net operating cost, total net operating cost, or total variable cost.

In [ ]:
def summary_value(row: pd.Series, column: str) -> float | None:
    value = pd.to_numeric(pd.Series([row.get(column)]), errors="coerce").iloc[0]
    return None if pd.isna(value) else float(value)


def net_cost_from_summary(summary: pd.DataFrame) -> pd.DataFrame:
    records = []
    for _, row in summary.iterrows():
        plant_name = row.get("plant_name", "unnamed plant")
        gross_cost = summary_value(row, "gross_operating_cost_EUR")
        variable_cost = summary_value(row, "total_variable_cost_EUR")
        net_incl_grid_fees = summary_value(
            row, "net_operating_cost_incl_grid_fees_EUR"
        )
        net_cost = summary_value(row, "net_operating_cost_EUR")
        total_net_cost = summary_value(row, "total_net_operating_cost_EUR")

        # Steel cost-minimisation has no market settlement columns: its
        # total_variable_cost_EUR is the complete process cost. Do not use
        # the zero-filled net-operating-cost placeholder in that case.
        if gross_cost is not None and abs(gross_cost) > 1e-9:
            if net_incl_grid_fees is not None:
                cost, basis = net_incl_grid_fees, "net operating cost incl. grid fees"
            elif net_cost is not None:
                cost, basis = net_cost, "net operating cost"
            else:
                cost, basis = gross_cost, "gross operating cost"
        elif total_net_cost is not None:
            cost, basis = total_net_cost, "total net operating cost"
        elif variable_cost is not None:
            cost, basis = variable_cost, "total variable cost"
        else:
            print(f"Skipping {plant_name}: no recognised cost column in summary_indicators.csv.")
            continue

        records.append({"plant_name": plant_name, "net_cost_EUR": cost, "cost_basis": basis})
    return pd.DataFrame(records)


OUTPUT_ROOT = RESULTS_DIR.parent.resolve()
summary_paths = sorted(OUTPUT_ROOT.glob("*/summary_indicators.csv"))

if not summary_paths:
    print(f"No summary_indicators.csv files found under {OUTPUT_ROOT}.")
else:
    cost_frames = []
    for summary_path in summary_paths:
        case_summary = pd.read_csv(summary_path)
        if "plant_name" not in case_summary.columns:
            print(f"Skipping {summary_path.parent.name}: no plant_name column.")
            continue

        case_costs = net_cost_from_summary(case_summary)
        if case_costs.empty:
            print(f"Skipping {summary_path.parent.name}: no recognised cost values.")
            continue

        case_costs.insert(0, "output_folder", summary_path.parent.name)
        cost_frames.append(case_costs)

    if not cost_frames:
        print("No recognised net-cost values were found in the output folders.")
    else:
        net_cost_by_plant = pd.concat(cost_frames, ignore_index=True)
        portfolio_total = net_cost_by_plant["net_cost_EUR"].sum()
        net_cost_by_plant.loc[len(net_cost_by_plant)] = {
            "output_folder": "All output folders",
            "plant_name": "All plants",
            "net_cost_EUR": portfolio_total,
            "cost_basis": "portfolio total",
        }
        net_cost_by_plant["net_cost_MEUR"] = net_cost_by_plant["net_cost_EUR"] / 1_000_000
        net_cost_by_plant["net_cost_BEUR"] = net_cost_by_plant["net_cost_EUR"] / 1_000_000_000
        display(
            net_cost_by_plant.style.format(
                {
                    "net_cost_EUR": "{:,.2f}",
                    "net_cost_MEUR": "{:,.2f}",
                    "net_cost_BEUR": "{:,.3f}",
                }
            )
        )


In [ ]:
from pathlib import Path

import pandas as pd

# Temporary CO2-cost correction for fuel emissions omitted by current plants.csv inputs.
# These provisional direct-combustion factors must later be replaced by explicit,
# literature-grounded coal_co2_factor and natural_gas_co2_factor input columns.
TEMP_COAL_CO2_FACTOR_T_PER_MWH = 0.34
TEMP_NATURAL_GAS_CO2_FACTOR_T_PER_MWH = 0.20

def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the FLEXIMOD project root from the current working directory."
    )


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "data" / "output"
INPUT_ROOT = PROJECT_ROOT / "data" / "input"


def input_case_for_output(output_folder: Path) -> Path | None:
    candidates = [
        case_dir
        for case_dir in INPUT_ROOT.iterdir()
        if case_dir.is_dir() and output_folder.name.startswith(f"{case_dir.name}_")
    ]
    return max(candidates, key=lambda path: len(path.name), default=None)


def temporary_summary_value(row: pd.Series, column: str) -> float | None:
    value = pd.to_numeric(pd.Series([row.get(column)]), errors="coerce").iloc[0]
    return None if pd.isna(value) else float(value)


def temporary_net_cost_from_summary(case_summary: pd.DataFrame) -> pd.DataFrame:
    records = []
    for _, row in case_summary.iterrows():
        plant_name = row.get("plant_name", "unnamed plant")
        gross_cost = temporary_summary_value(row, "gross_operating_cost_EUR")
        variable_cost = temporary_summary_value(row, "total_variable_cost_EUR")
        net_incl_grid_fees = temporary_summary_value(
            row, "net_operating_cost_incl_grid_fees_EUR"
        )
        net_cost = temporary_summary_value(row, "net_operating_cost_EUR")
        total_net_cost = temporary_summary_value(row, "total_net_operating_cost_EUR")

        if gross_cost is not None and abs(gross_cost) > 1e-9:
            cost = (
                net_incl_grid_fees
                if net_incl_grid_fees is not None
                else net_cost if net_cost is not None else gross_cost
            )
        elif total_net_cost is not None:
            cost = total_net_cost
        elif variable_cost is not None:
            cost = variable_cost
        else:
            continue
        records.append({"plant_name": plant_name, "original_net_cost_EUR": cost})
    return pd.DataFrame(records)


original_cost_frames = []
for summary_path in sorted(OUTPUT_ROOT.glob("*/summary_indicators.csv")):
    case_costs = temporary_net_cost_from_summary(pd.read_csv(summary_path))
    if case_costs.empty:
        continue
    case_costs.insert(0, "output_folder", summary_path.parent.name)
    original_cost_frames.append(case_costs)

original_costs = (
    pd.concat(original_cost_frames, ignore_index=True)
    if original_cost_frames
    else pd.DataFrame(columns=["output_folder", "plant_name", "original_net_cost_EUR"])
)

correction_records = []
for output_folder in sorted(path for path in OUTPUT_ROOT.iterdir() if path.is_dir()):
    dispatch_path = output_folder / "dispatch_results.csv"
    input_case = input_case_for_output(output_folder)
    if not dispatch_path.exists() or input_case is None:
        continue

    forecasts_path = input_case / "forecasts_df.csv"
    if not forecasts_path.exists():
        print(f"Skipping {output_folder.name}: forecasts_df.csv was not found.")
        continue

    dispatch_columns = [
        "datetime",
        "plant_name",
        "steel_route",
        "coal_consumption_MWh",
        "natural_gas_consumption_MWh",
    ]
    try:
        case_dispatch = pd.read_csv(dispatch_path, usecols=dispatch_columns)
        case_forecasts = pd.read_csv(
            forecasts_path,
            usecols=["datetime", "co2_price"],
        )
    except ValueError as exc:
        print(f"Skipping {output_folder.name}: {exc}")
        continue

    case_dispatch["datetime"] = pd.to_datetime(case_dispatch["datetime"], errors="raise")
    case_forecasts["datetime"] = pd.to_datetime(case_forecasts["datetime"], errors="raise")
    case_forecasts["co2_price"] = pd.to_numeric(
        case_forecasts["co2_price"], errors="raise"
    )
    case_forecasts = case_forecasts.drop_duplicates("datetime")
    corrected = case_dispatch.merge(
        case_forecasts,
        on="datetime",
        how="left",
        validate="many_to_one",
    )
    if corrected["co2_price"].isna().any():
        missing = int(corrected["co2_price"].isna().sum())
        raise ValueError(
            f"{output_folder.name}: {missing} dispatch rows have no matching CO2 price"
        )

    coal = pd.to_numeric(corrected["coal_consumption_MWh"], errors="coerce").fillna(0.0)
    natural_gas = pd.to_numeric(
        corrected["natural_gas_consumption_MWh"], errors="coerce"
    ).fillna(0.0)
    route = corrected["steel_route"].astype(str).str.lower()

    # DRI coal currently defaults to a zero factor. DRI natural gas is not
    # corrected here because its existing 0.5 tCO2/MWh default is already in net cost.
    # Integrated BF-BOF currently defaults both coal and natural-gas factors to zero.
    corrected["temporary_missing_co2_emissions_t"] = 0.0
    dri_route = route.isin(["dri_eaf", "dri_bof"])
    bf_bof_route = route.eq("bf_bof")
    corrected.loc[dri_route, "temporary_missing_co2_emissions_t"] = (
        coal[dri_route] * TEMP_COAL_CO2_FACTOR_T_PER_MWH
    )
    corrected.loc[bf_bof_route, "temporary_missing_co2_emissions_t"] = (
        coal[bf_bof_route] * TEMP_COAL_CO2_FACTOR_T_PER_MWH
        + natural_gas[bf_bof_route] * TEMP_NATURAL_GAS_CO2_FACTOR_T_PER_MWH
    )
    corrected["temporary_missing_co2_cost_EUR"] = (
        corrected["temporary_missing_co2_emissions_t"] * corrected["co2_price"]
    )

    per_plant_correction = corrected.groupby(
        ["plant_name", "steel_route"], as_index=False, dropna=False
    ).agg(
        temporary_missing_co2_emissions_t=(
            "temporary_missing_co2_emissions_t",
            "sum",
        ),
        temporary_missing_co2_cost_EUR=("temporary_missing_co2_cost_EUR", "sum"),
    )
    per_plant_correction.insert(0, "output_folder", output_folder.name)
    correction_records.append(per_plant_correction)

if not correction_records:
    print("No completed steel dispatch outputs were found for CO2-cost correction.")
else:
    temporary_co2_correction = pd.concat(correction_records, ignore_index=True)
    adjusted_net_cost_by_plant = original_costs.merge(
        temporary_co2_correction,
        on=["output_folder", "plant_name"],
        how="inner",
        validate="one_to_one",
    )
    adjusted_net_cost_by_plant["adjusted_net_cost_EUR"] = (
        adjusted_net_cost_by_plant["original_net_cost_EUR"]
        + adjusted_net_cost_by_plant["temporary_missing_co2_cost_EUR"]
    )
    adjusted_net_cost_by_plant["adjusted_net_cost_MEUR"] = (
        adjusted_net_cost_by_plant["adjusted_net_cost_EUR"] / 1_000_000
    )
    adjusted_net_cost_by_plant["adjusted_net_cost_BEUR"] = (
        adjusted_net_cost_by_plant["adjusted_net_cost_EUR"] / 1_000_000_000
    )

    total_columns = [
        "original_net_cost_EUR",
        "temporary_missing_co2_emissions_t",
        "temporary_missing_co2_cost_EUR",
        "adjusted_net_cost_EUR",
        "adjusted_net_cost_MEUR",
        "adjusted_net_cost_BEUR",
    ]
    total_row = {
        "output_folder": "All output folders",
        "plant_name": "All plants",
        "steel_route": "portfolio total",
        **{column: adjusted_net_cost_by_plant[column].sum() for column in total_columns},
    }
    adjusted_net_cost_by_plant.loc[len(adjusted_net_cost_by_plant)] = total_row
    display(
        adjusted_net_cost_by_plant.style.format(
            {
                "original_net_cost_EUR": "{:,.2f}",
                "temporary_missing_co2_emissions_t": "{:,.2f}",
                "temporary_missing_co2_cost_EUR": "{:,.2f}",
                "adjusted_net_cost_EUR": "{:,.2f}",
                "adjusted_net_cost_MEUR": "{:,.2f}",
                "adjusted_net_cost_BEUR": "{:,.3f}",
            }
        )
    )


In [ ]:
# Energy demand by carrier for every plant in every completed output folder.
# The aggregate dispatch columns are used to avoid double-counting technology-level flows.
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_fleximod_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the FLEXIMOD project root.")


PROJECT_ROOT_ENERGY = find_fleximod_root()
OUTPUT_ROOT_ENERGY = PROJECT_ROOT_ENERGY / "data" / "output"

# Authoritative aggregate columns and their physical carrier units.
CARRIER_COLUMNS = {
    "Electricity": ("total_electricity_consumption_MWh", "MWh_el"),
    "Coal": ("coal_consumption_MWh", "MWh_fuel"),
    "Natural gas": ("natural_gas_consumption_MWh", "MWh_fuel"),
    "Hydrogen": ("hydrogen_consumption_MWh", "MWh_H2"),
}

energy_records = []
dispatch_files = sorted(OUTPUT_ROOT_ENERGY.glob("*/dispatch_results.csv"))

for dispatch_file in dispatch_files:
    available_columns = set(pd.read_csv(dispatch_file, nrows=0).columns)
    carrier_columns = {
        carrier: (column, unit)
        for carrier, (column, unit) in CARRIER_COLUMNS.items()
        if column in available_columns
    }
    if "plant_name" not in available_columns or not carrier_columns:
        continue

    identity_columns = ["plant_name"]
    for optional_column in ("plant_type", "steel_route"):
        if optional_column in available_columns:
            identity_columns.append(optional_column)

    usecols = identity_columns + [column for column, _ in carrier_columns.values()]
    data = pd.read_csv(dispatch_file, usecols=usecols)
    for column, _ in carrier_columns.values():
        data[column] = pd.to_numeric(data[column], errors="coerce").fillna(0.0)

    grouped = data.groupby(identity_columns, dropna=False, as_index=False).sum(numeric_only=True)
    for _, plant in grouped.iterrows():
        for carrier, (column, unit) in carrier_columns.items():
            energy_records.append(
                {
                    "output_folder": dispatch_file.parent.name,
                    "plant_name": plant["plant_name"],
                    "plant_type": plant.get("plant_type", ""),
                    "steel_route": plant.get("steel_route", ""),
                    "energy_carrier": carrier,
                    "carrier_unit": unit,
                    "energy_demand_MWh": float(plant[column]),
                }
            )

energy_demand_by_carrier = pd.DataFrame(energy_records)

if energy_demand_by_carrier.empty:
    print(f"No dispatch results containing supported energy-carrier columns found under {OUTPUT_ROOT_ENERGY}")
else:
    energy_demand_by_carrier["energy_demand_GWh"] = (
        energy_demand_by_carrier["energy_demand_MWh"] / 1_000.0
    )
    plant_totals = energy_demand_by_carrier.groupby(
        ["output_folder", "plant_name"], dropna=False
    )["energy_demand_MWh"].transform("sum")
    energy_demand_by_carrier["carrier_share_percent"] = (
        energy_demand_by_carrier["energy_demand_MWh"]
        .div(plant_totals.where(plant_totals.ne(0)))
        .mul(100.0)
        .fillna(0.0)
    )

    energy_demand_by_carrier = energy_demand_by_carrier.sort_values(
        ["output_folder", "plant_name", "energy_carrier"]
    ).reset_index(drop=True)

    print(
        f"Energy demand by carrier: {energy_demand_by_carrier['plant_name'].nunique()} unique plant name(s) "
        f"across {energy_demand_by_carrier['output_folder'].nunique()} completed output folder(s)."
    )
    display(
        energy_demand_by_carrier.style.format(
            {
                "energy_demand_MWh": "{:,.2f}",
                "energy_demand_GWh": "{:,.3f}",
                "carrier_share_percent": "{:,.2f}%",
            }
        )
    )

    # Compact plant-by-carrier view; values are GWh over the simulated period.
    energy_demand_pivot_GWh = (
        energy_demand_by_carrier.pivot_table(
            index=["output_folder", "plant_name", "steel_route"],
            columns="energy_carrier",
            values="energy_demand_GWh",
            aggfunc="sum",
            fill_value=0.0,
        )
        .reset_index()
        .rename_axis(columns=None)
    )
    carrier_value_columns = [
        column for column in CARRIER_COLUMNS if column in energy_demand_pivot_GWh.columns
    ]
    energy_demand_pivot_GWh["Total final energy"] = energy_demand_pivot_GWh[
        carrier_value_columns
    ].sum(axis=1)
    display(energy_demand_pivot_GWh.style.format({column: "{:,.3f}" for column in carrier_value_columns + ["Total final energy"]}))
